In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout, SimpleRNN, LSTM, LayerNormalization, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import re
import scipy.io
import psutil
import os
import math
import cmath 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2


#infile_path = 'post_total_data.csv'
infile_path = 'post_dist_bb_1GHZ_csv.csv'
df = pd.read_csv(infile_path)

xin_real = df['xI_real']
xin_imag = df['xI_imag']
yout_real = df['yout_real']
yout_imag = df['yout_imag']

t_ns = np.arange(1,30001)

# read real, imaginary parts and calculate amplitude
xin = xin_real + 1j*xin_imag
xin_amp = abs(xin)
#xin_phase = phase(xin)
xinn = xin/max(xin_amp)
xinn_amp = abs(xinn)

yout = yout_real + 1j*yout_imag
yout_amp = abs(yout)
# normalize output to peak value
youtn = yout/max(yout_amp)
youtn_amp = abs(youtn)
#yout_phase = cmath.phase(y_out)

# compute rms value for each signal
xin_rms = np.sqrt(np.mean(np.abs(xin)**2))
yout_rms = np.sqrt(np.mean(np.abs(yout)**2))
gainrms_dB = 20*math.log10(yout_rms/xin_rms)
print(gainrms_dB)

y_arr = np.column_stack((xin_real, xin_imag))
youtn = np.array(youtn)
youtn_real = youtn.real
youtn_imag = youtn.imag
X_arr = np.column_stack((yout_real, yout_imag))

y = pd.DataFrame(X_arr, columns=['yout_real', 'yout_imag'])
X = pd.DataFrame(y_arr, columns=['xin_real', 'xin_imag'])


#Old sequences
N_train = 24_000

X_new = y.iloc[N_train:]
#y_new = y.iloc[240000:300001]
print("Shape of X_new:", X_new.shape)
# Keep the first 90,000 rows in both dataframes:
X = X.iloc[:N_train]
y = y.iloc[:N_train]


print("Shape of y:", y.shape)

1.325097626610605
Shape of X_new: (6000, 2)
Shape of y: (24000, 2)


In [6]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Build the model
model = Sequential([
    Input(shape=(2,)),  # Explicitly defining the input shape
    Dense(32, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(2)  # Output layer with 2 neurons for xI_real and xI_imag
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, validation_split=0.2, verbose=1)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)


y_pred = model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f'Test Loss: {test_loss}')
print(f'Test accuracy: {test_accuracy}')
print(f'Test RMSE: {rmse}')

print("Training complete")

# Function to parse complex numbers, including those with scientific notation from excel csv file
def parse_complex_number(s):
    s = s.replace("i", "j")  
    # Updated regex to handle scientific notation
    match = re.match(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([-+]\d*\.?\d+(?:[eE][-+]?\d+)?)j", s.replace(" ", ""))
    if match:
        real_part = float(match.group(1))
        imag_part = float(match.group(2))
        return real_part, imag_part
    else:
        raise ValueError(f"Malformed complex number: {s}")


# # Loop through files numbered 1 to 31
for i in range(1, 52):
    # Change this to working directory 
    # Load the CSV file for predictions\\
    predict_file_path = rf'C:\Users\Luke McCubbin\Box\NN PA DPD\Postdistortion_study_UCSD_10_5_2025\sinewaves\sine_test_{i}.csv'
    df_predict = pd.read_csv(predict_file_path)

    # Separate the real and imaginary parts of the 'yout' complex numbers
    df_predict[['xI_real', 'xI_imag']] = df_predict['xI'].apply(
        lambda x: pd.Series(parse_complex_number(x))
    )

    # Prepare input features for prediction
    X_new = df_predict[['xI_real', 'xI_imag']]

    # Predict using the model
    y_new_pred = model.predict(X_new)

    # Store predictions back into the DataFrame
    df_predict['yout_predict_real'] = y_new_pred[:, 0]
    df_predict['yout_predict_imag'] = y_new_pred[:, 1]

    # Combine real and imaginary parts into a complex number
    df_predict['yout_predict'] = (df_predict['yout_predict_real'] + 1j * df_predict['yout_predict_imag']).astype(np.complex128)

    # Prepare data for saving as MAT file
    data_to_save = {'yout_predict': df_predict['yout_predict'].values}

    # Change this to directory where you want results to be saved. Might have to create a folder called sineresults
    
    # Save to a MAT file with a dynamic filename
    output_mat_file_path = rf'C:\Users\Luke McCubbin\Box\NN PA DPD\Postdistortion_study_UCSD_10_5_2025\sineresults\sine_result_{i}.mat'
    scipy.io.savemat(output_mat_file_path, data_to_save)

    print(f'Predictions saved to {output_mat_file_path}')

Epoch 1/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9421 - loss: 0.0247 - val_accuracy: 0.9617 - val_loss: 0.0040
Epoch 2/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9586 - loss: 0.0041 - val_accuracy: 0.9625 - val_loss: 0.0039
Epoch 3/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9604 - loss: 0.0039 - val_accuracy: 0.9617 - val_loss: 0.0040
Epoch 4/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9583 - loss: 0.0040 - val_accuracy: 0.9581 - val_loss: 0.0042
Epoch 5/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9604 - loss: 0.0039 - val_accuracy: 0.9620 - val_loss: 0.0041
Epoch 6/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9576 - loss: 0.0039 - val_accuracy: 0.9628 - val_loss: 0.0040
Epoch 7/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9592 - loss: 0.0038 - val_accuracy: 0.9628 - val_loss: 0.0039
Epoch 8/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9585 - loss: 0.0039 - val_accuracy: 0.